In [ ]:
# Cell 1 — #012 setup + data: US500 30m close-stamped panel (reused from #013 build).
# 1-min Dukascopy BID closes -> 30m bars, label="right" stamps by CLOSE time (PIT-honest).
# check_outliers=False: COVID March-2020 moves reviewed and accepted in #013 Cell 1.
# Verify the three ET stamps #012 needs exist daily: 10:00 (signal), 15:30 (entry), 16:00 (exit).
import numpy as np
import pandas as pd
from backtest.data import DataPanel

RAW = pd.read_parquet(r"C:\Users\User\backtest_engine\backtest_engine2\data\us500_1m.parquet")
s = RAW.set_index("open_time")["close"].sort_index()
s.index = s.index.tz_localize(None)  # tz-naive UTC

px30 = s.resample("30min", closed="left", label="right").last().dropna()
prices = px30.to_frame("USA500")
panel = DataPanel(prices, check_outliers=False)

ET = "America/New_York"


def _et(dates):
    return dates.tz_localize("UTC").tz_convert(ET)


print(f"Bars: {len(panel.dates)} | range {panel.dates[0]} -> {panel.dates[-1]}")

et_idx = _et(prices.index)
px_ = prices["USA500"]
weekday = pd.Series(et_idx.dayofweek, index=prices.index)
for hh, mm, label in [(10, 0, "10:00 signal"), (15, 30, "15:30 entry"), (16, 0, "16:00 exit")]:
    mask = (et_idx.hour == hh) & (et_idx.minute == mm) & (weekday < 5).values
    days = pd.Series(et_idx[mask].date).nunique()
    print(f"  {label}: {int(mask.sum())} stamps on {days} weekdays")

# Trading-day alignment: how many days have ALL THREE stamps (a complete signal->trade day)?
df = pd.DataFrame({"d": et_idx.date, "h": et_idx.hour, "m": et_idx.minute})
have = df.groupby("d").apply(
    lambda g: ((g.h == 10) & (g.m == 0)).any()
    and ((g.h == 15) & (g.m == 30)).any()
    and ((g.h == 16) & (g.m == 0)).any()
)
print(f"Complete days (10:00 + 15:30 + 16:00 ET): {int(have.sum())} of {len(have)}")
yrs = pd.Series(pd.to_datetime(have[have].index)).dt.year.value_counts().sort_index()
print("Complete days per year:", yrs.to_dict())


In [ ]:
# Cell 2 — #012 decay check FIRST (panel-level, no engine): does the first half-hour
# still predict the last half-hour, 2019-2026 (all post-publication)?
# t1: signal = prior 16:00 close -> 10:00 (gap incl., paper headline predictor)
# t2: signal = 9:30 -> 10:00 (gap excl.)
# target: 15:30 -> 16:00 return; strategy return = sign(signal) * target.
def _stamp(hh, mm):
    m = (et_idx.hour == hh) & (et_idx.minute == mm)
    out = px_[m]
    out.index = pd.DatetimeIndex(et_idx[m].date)  # key by ET trading day
    return out

p0930, p1000 = _stamp(9, 30), _stamp(10, 0)
p1530, p1600 = _stamp(15, 30), _stamp(16, 0)

days = p1000.index.intersection(p1530.index).intersection(p1600.index)
prev_close = p1600.reindex(days).shift(1)  # prior trading day's close

sig1 = (p1000.reindex(days) / prev_close - 1).dropna()                 # t1: gap incl.
sig2 = (p1000.reindex(days) / p0930.reindex(days) - 1).dropna()        # t2: gap excl.
tgt = (p1600.reindex(days) / p1530.reindex(days) - 1)

for name, sig in [("t1 gap-incl (paper)", sig1), ("t2 first-30m only", sig2)]:
    r = (np.sign(sig) * tgt.reindex(sig.index)).dropna()
    r = r[np.sign(sig.reindex(r.index)) != 0]
    sh = r.mean() / r.std() * np.sqrt(250)
    print(f"{name}: {len(r)} days | hit {100 * (r > 0).mean():.1f}% | "
          f"gross {1e4 * r.mean():+.2f} bps/trade | Sharpe {sh:+.2f}")
print("paper (1993-2013): ~+3-7 bps/trade | cost gate ~2.0 bps/round trip\n")

print(f"{'year':>5} | {'t1 bps':>7} {'t1 hit':>6} {'t1 Sh':>6} | {'t2 bps':>7} {'t2 hit':>6} {'t2 Sh':>6}")
for yr in range(2019, 2027):
    row = []
    for sig in (sig1, sig2):
        r = (np.sign(sig) * tgt.reindex(sig.index)).dropna()
        r = r[r.index.year == yr]
        if len(r) < 30:
            row.append(None)
            continue
        row.append((1e4 * r.mean(), 100 * (r > 0).mean(),
                    r.mean() / r.std() * np.sqrt(250)))
    f = lambda c: f"{c[0]:>+7.2f} {c[1]:>6.1f} {c[2]:>+6.2f}" if c else " " * 21
    print(f"{yr:>5} | {f(row[0])} | {f(row[1])}")


In [ ]:
# Cell 3 — #012 VERDICT: KILLED at the decay check (pre-registered rule fired).
# Rule: gross < 2bp gate on both pre-registered trials in 2019-2026 -> kill, no variants.
# Neither trial shows ANY gross edge: signs flip year-to-year (noise), no decay trend,
# no stable inversion. Never reached the strategy/costs/engine stages. K=2; DSR moot
# (negative realized gross). The 2014-public/2018-JFE anomaly is absent from its home
# index in the entire post-publication sample we can observe.
print("#012 index intraday momentum (US500): KILLED — no gross signal post-publication")
print("  t1 gap-incl (paper): 1852 days | hit 49.4% | -0.14 bps/trade | Sh -0.07")
print("  t2 first-30m only:   1849 days | hit 48.7% | -0.57 bps/trade | Sh -0.30")
print("  best year (t1 2020 +2.45bp) barely reaches the 2bp gate once in 8 years")
print("  queue state: #014/#015 swap-gated (USER ACTION: FTMO swap points), #016 last")
